# Cardiac JEPA — pré-entraînement sur Colab

Entraîne le JEPA latent sur PTB-XL (100 Hz) avec le GPU Colab.

**Avant de lancer** : `Exécution ▸ Modifier le type d'exécution ▸ GPU`.

Ce notebook est *idempotent* : re-lance-le après une déconnexion, il reprend
l'entraînement là où il s'était arrêté (`--resume auto`).

**Une seule chose à renseigner** : `REPO_URL` dans la cellule de configuration.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

## 2. Monter Drive et configurer les chemins

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# >>> RENSEIGNE TON URL DE REPO ICI (pas de chevrons !) <<<
# exemple : "https://github.com/moncompte/cardiac-jepa.git"
REPO_URL = "https://github.com/JulesV19/cardiac-JEPA"

if not REPO_URL:
    raise ValueError(
        "REPO_URL est vide. Mets l'URL de ton repo GitHub dans cette cellule, "
        "par ex. https://github.com/moncompte/cardiac-jepa.git")

DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                            # les 2 CSV (petits)
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                 # cache signaux (1,05 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')               # copie sur disque local = rapide
RUN_DIR     = DRIVE_ROOT / 'runs' / 'tiny_v1'                # checkpoints + metrics.csv
REPO_DIR    = Path('/content/cardiac-jepa')

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Drive prêt :", DRIVE_ROOT)

## 3. Cloner / mettre à jour le code

Tu édites en local dans VSCode, tu `git push`, puis tu ré-exécutes cette cellule.

In [ ]:
import subprocess

if REPO_DIR.exists():
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError(
        f"git a échoué (code {r.returncode}). Vérifie REPO_URL, que le repo existe, "
        "et qu'il est public (sinon il faut un token d'accès).")

assert (REPO_DIR / "requirements.txt").exists(), \
    f"{REPO_DIR}/requirements.txt introuvable — le clone a-t-il vraiment réussi ?"

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("code prêt :", REPO_DIR)

## 4. Données (déposées manuellement sur Drive)

Le cache est construit en local et déposé sur Drive — **rien à télécharger ici**.
Arborescence attendue :

```
MyDrive/cardiac-jepa/
├── ptbxl_100hz.npy        (1,05 Go)  cache des signaux 100 Hz
└── data/
    ├── ptbxl_database.csv (6,6 Mo)
    └── scp_statements.csv (10 Ko)
```

Pourquoi un cache plutôt que les fichiers bruts : `records100` contient **~43 000 petits
fichiers**, atroces à uploader sur Drive *et* atroces à lire depuis un Drive monté.
Un seul fichier contigu, copié sur le disque local de Colab, c'est instantané.

L'ordre des lignes de `ptbxl_database.csv` définit l'index du cache : **ces deux fichiers
doivent aller ensemble.**

In [ ]:
import shutil, time
import numpy as np

required = [CACHE_DRIVE,
            DATA_DRIVE / 'ptbxl_database.csv',
            DATA_DRIVE / 'scp_statements.csv']
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants sur Drive :\n  " + "\n  ".join(str(m) for m in missing) +
        "\n\nDépose-les exactement à ces emplacements, puis relance cette cellule.")

if not CACHE_LOCAL.exists():
    print("Copie du cache Drive -> disque local (~1 Go, une fois par session)...")
    t0 = time.time()
    shutil.copy(CACHE_DRIVE, CACHE_LOCAL)
    print(f"copié en {time.time()-t0:.0f}s")

# Le code lit ces variables d'environnement (voir jepa/data.py).
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)   # CSV uniquement
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

a = np.load(CACHE_LOCAL, mmap_mode='r')
assert a.shape[1:] == (1000, 12), f"forme inattendue: {a.shape}"
print(f"cache OK : {a.shape} {a.dtype}")

## 5. Sanity check : les données se chargent

In [ ]:
from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain')
x = ds[0]
print(f"pretrain={len(ds)} ECG | un échantillon: {tuple(x.shape)} "
      f"| moyenne={x.mean():.3f} std={x.std():.3f}  (z-normé par dérivation)")

## 6. Entraînement

`--resume auto` reprend depuis `latest.pt` s'il existe. Si Colab te déconnecte,
ré-exécute simplement cette cellule : tu perds au plus une epoch.

In [ ]:
!python -m jepa.train \
    --config jepa/configs/tiny.yaml \
    --out "{RUN_DIR}" \
    --resume auto \
    --workers 2

## 7. Courbes de collapse

Le critère de succès de cette phase. À surveiller :
- `emb_std_ctx` **ne doit pas tendre vers 0**,
- `eff_rank_ctx` doit rester élevé,
- `jepa` doit baisser *progressivement* (pas s'effondrer à ~0 d'emblée),
- `eff_rank_tgt` : point de vigilance connu — la VICReg ne régularise que le contexte.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

m = pd.read_csv(RUN_DIR / 'metrics.csv')
tr, va = m[m.phase == 'train'], m[m.phase == 'val']

fig, ax = plt.subplots(1, 4, figsize=(19, 4))

# LE graphe qui compte : qualité de prédiction, insensible à la difficulté des cibles.
ax[0].plot(va.epoch, va.r2, marker='o', color='#C44E52', label='R²')
ax[0].axhline(0, ls='--', c='k', lw=1, label='prédicteur trivial (moyenne)')
ax[0].set_title('R² — apprend-on vraiment ?'); ax[0].set_xlabel('epoch'); ax[0].legend()

ax[1].plot(tr.step, tr.jepa, lw=0.8)
ax[1].set_title('L_jepa (cible mouvante : non comparable seule)'); ax[1].set_xlabel('step')

ax[2].plot(va.epoch, va.emb_std_ctx, label='contexte')
ax[2].plot(va.epoch, va.emb_std_tgt, label='cible')
ax[2].plot(va.epoch, va.pred_std, label='prédiction')
ax[2].axhline(0.1, ls='--', c='r', lw=1, label='seuil collapse')
ax[2].set_title('écart-type des descriptions'); ax[2].set_xlabel('epoch'); ax[2].legend()

ax[3].plot(va.epoch, va.eff_rank_ctx, label='contexte')
ax[3].plot(va.epoch, va.eff_rank_tgt, label='cible')
ax[3].set_title('rang effectif (max 192)'); ax[3].set_xlabel('epoch'); ax[3].legend()

plt.tight_layout(); plt.show()
print(va[['epoch','r2','cos','emb_std_ctx','eff_rank_ctx','eff_rank_tgt']].tail(10).to_string(index=False))

## 8. Sonde linéaire — la vraie mesure de qualité

Le R² est un diagnostic, pas l'objectif. Seule la sonde dit si les représentations
valent quelque chose : encodeur cible EMA **gelé** → moyenne des 480 tokens → 192 features
→ une couche linéaire → 5 superclasses. Métrique : **macro-AUROC sur le fold 10** (jamais
utilisé jusqu'ici).

**On lance d'abord la baseline de contrôle** : le même protocole sur un encodeur aux poids
**aléatoires**. Si le JEPA ne la bat pas nettement, il n'a rien appris d'utile.

In [ ]:
# Baseline de contrôle : encodeur NON entraîné.
!python -m jepa.probe --random-init --workers 2 --out "{RUN_DIR}/probe_random.json"

In [ ]:
# Sonde sur le JEPA pré-entraîné (dernier checkpoint : 100 epochs -> ckpt_e99.pt).
CKPT = RUN_DIR / 'ckpt_e99.pt'
assert CKPT.exists(), f"{CKPT} absent — liste: {sorted(p.name for p in RUN_DIR.glob('*.pt'))}"
!python -m jepa.probe --ckpt "{CKPT}" --encoder target --workers 2 --out "{RUN_DIR}/probe_jepa.json"

## 9. Ablation `lambda_cov`

Hypothèse : la pénalité de covariance blanchit trop les représentations (rang 175/192)
et plafonne la prédictibilité. On relance à l'identique avec `lambda_cov: 0.04 → 0.005`.
**Une seule variable change.**

On compare ensuite les **AUROC**, pas les R².

In [ ]:
RUN_ABL = DRIVE_ROOT / 'runs' / 'tiny_lowcov'
RUN_ABL.mkdir(parents=True, exist_ok=True)

!python -m jepa.train \
    --config jepa/configs/tiny_lowcov.yaml \
    --out "{RUN_ABL}" \
    --resume auto \
    --workers 2

In [ ]:
CKPT_ABL = RUN_ABL / 'ckpt_e99.pt'
assert CKPT_ABL.exists(), f"{CKPT_ABL} absent"
!python -m jepa.probe --ckpt "{CKPT_ABL}" --encoder target --workers 2 --out "{RUN_ABL}/probe_jepa.json"

### Comparatif final

In [ ]:
import json, pandas as pd

rows = []
for name, path in [("random (contrôle)", RUN_DIR/'probe_random.json'),
                   ("JEPA lambda_cov=0.04", RUN_DIR/'probe_jepa.json'),
                   ("JEPA lambda_cov=0.005", RUN_ABL/'probe_jepa.json')]:
    if path.exists():
        r = json.load(open(path))
        rows.append({"modèle": name, "macro-AUROC (test)": round(r["test_macro_auroc"], 4),
                     **{k: round(v, 3) for k, v in r["test_per_class"].items()}})
print(pd.DataFrame(rows).to_string(index=False))
print("\nRéférence : un ResNet supervisé atteint ~0.93 de macro-AUROC sur ce benchmark.")

## 10. Phase 2 — le classifieur (fine-tuning)

Encodeur JEPA **dégelé**, `lr` 10x plus faible que la tête. Sélection du meilleur epoch
sur le fold 9 (`best.pt` réécrit dès amélioration), test sur le fold 10 **une seule fois**.

On lance **d'abord le contrôle** : le même fine-tuning depuis un encodeur *aléatoire*.
Sans lui, un AUROC de 0,88 ne prouve rien — un ViT-tiny fine-tuné from scratch pourrait
l'atteindre seul, et le pré-entraînement n'aurait servi à rien.

In [ ]:
CLF_RANDOM = DRIVE_ROOT / 'runs' / 'clf_random'
!python -m jepa.classify --random-init --out "{CLF_RANDOM}" --workers 2

In [ ]:
CKPT = RUN_DIR / 'ckpt_e99.pt'
assert CKPT.exists(), sorted(p.name for p in RUN_DIR.glob('*.pt'))
CLF_JEPA = DRIVE_ROOT / 'runs' / 'clf_jepa'
!python -m jepa.classify --ckpt "{CKPT}" --encoder target --out "{CLF_JEPA}" --workers 2

### Le tableau qui conclut la phase 2

In [ ]:
import json, pandas as pd

rows = []
srcs = [("sonde  — encodeur aléatoire gelé",  RUN_DIR / 'probe_random.json'),
        ("sonde  — JEPA gelé",                RUN_DIR / 'probe_jepa.json'),
        ("fine-tuning — depuis aléatoire",    CLF_RANDOM / 'result.json'),
        ("fine-tuning — depuis JEPA",         CLF_JEPA / 'result.json')]
for name, p in srcs:
    if p.exists():
        r = json.load(open(p))
        rows.append({"modèle": name, "macro-AUROC": round(r['test_macro_auroc'], 4),
                     **{k: round(v, 3) for k, v in r['test_per_class'].items()}})
print(pd.DataFrame(rows).to_string(index=False))
print("\nRéférence littérature (ResNet supervisé, non reproduit ici) : ~0.93")
print("\nLa question : 'fine-tuning depuis JEPA' bat-il 'fine-tuning depuis aléatoire' ?")
print("Si l'écart est < 0.01, il est indistinguable du bruit de seed (aucune graine fixée).")

## 11. Régime peu-de-labels — là où le SSL doit prouver sa valeur

À 100 % des labels, le fine-tuning efface 85 % de l'avantage JEPA (écart final +0,009,
non concluant). C'est le régime où l'auto-supervision sert le moins.

La vraie question : **avec peu de labels, l'écart tient-il ?** On fine-tune JEPA vs
aléatoire sur 1 %, 5 %, 10 %, 100 % des labels. Sous-échantillon identique entre les deux
bras (même graine), val et test complets.

Les petites fractions sont **bruitées** (à 1 %, ~171 ECG d'entraînement, HYP ~24 positifs) :
on les rejoue sur **5 graines** pour des barres d'erreur, l'écart étant *pairé* par graine.
10 % et 100 % sont solides → une seule graine.

La cellule est **idempotente** : elle saute les runs déjà faits (dont l'ancien s0), donc
relançable après une déconnexion. Compter ~1-2 h.

In [ ]:
import subprocess, sys, json
from pathlib import Path

CKPT = RUN_DIR / 'ckpt_e99.pt'
assert CKPT.exists(), sorted(p.name for p in RUN_DIR.glob('*.pt'))
SWEEP = DRIVE_ROOT / 'runs' / 'lowlabel'

# Petites fractions bruitées -> plusieurs graines. 10 %/100 % solides -> une seule.
# (l'ancien run s0 pour toutes les fractions est réutilisé tel quel : rétrocompatible.)
SEEDS_BY_FRAC = {0.01: [0, 1, 2, 3, 4],
                 0.05: [0, 1, 2, 3, 4],
                 0.10: [0],
                 1.00: [0]}
FRACS = sorted(SEEDS_BY_FRAC)

def run(arm, frac, seed):
    out = SWEEP / f'{arm}_f{frac}_s{seed}'
    if (out / 'result.json').exists():
        print(f'  {arm} frac={frac} seed={seed} : déjà fait'); return
    src = ['--random-init'] if arm == 'random' else ['--ckpt', str(CKPT), '--encoder', 'target']
    cmd = [sys.executable, '-m', 'jepa.classify', *src, '--out', str(out),
           '--train-frac', str(frac), '--seed', str(seed), '--workers', '2']
    print(f'\n=== {arm}  frac={frac}  seed={seed} ===', flush=True)
    assert subprocess.run(cmd).returncode == 0, f'échec {arm} frac={frac} seed={seed}'

for frac in FRACS:
    for seed in SEEDS_BY_FRAC[frac]:
        for arm in ('random', 'jepa'):
            run(arm, frac, seed)
print('\nsweep terminé')

### Courbe peu-de-labels

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt

rows = []
for frac in FRACS:
    for seed in SEEDS_BY_FRAC[frac]:
        res = {}
        for arm in ('random', 'jepa'):
            p = SWEEP / f'{arm}_f{frac}_s{seed}' / 'result.json'
            if p.exists():
                res[arm] = json.load(open(p))['test_macro_auroc']
        if len(res) == 2:
            rows.append({'frac': frac, 'seed': seed, 'random': res['random'],
                         'jepa': res['jepa'], 'gap': res['jepa'] - res['random']})
d = pd.DataFrame(rows)

# Écart PAIRÉ par graine (même sous-échantillon entre bras) = le bon estimateur de l'effet.
def agg(g):
    return pd.Series({'random': g.random.mean(), 'jepa': g.jepa.mean(),
                      'écart': g.gap.mean(),
                      'écart_std': g.gap.std(ddof=1) if len(g) > 1 else 0.0,
                      'n': len(g)})
tab = d.groupby('frac').apply(agg)
print(tab.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
for arm, c in [('jepa', '#C44E52'), ('random', '#4C72B0')]:
    m = d.groupby('frac')[arm].mean()
    s = d.groupby('frac')[arm].std(ddof=1).fillna(0.0)
    ax.errorbar(m.index * 100, m.values, yerr=s.values, marker='o', capsize=3,
                color=c, label=arm)
ax.set_xscale('log')
ax.set_xlabel("% des labels d'entraînement (échelle log)")
ax.set_ylabel('macro-AUROC (test)')
ax.set_title('Le pré-entraînement JEPA aide-t-il quand les labels manquent ?')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 12. Décodeur de signal — le JEPA récupère-t-il le tracé sous le masque ?

Le JEPA est *latent* : il prédit des embeddings, jamais des mV. Question : quand le predictor
devine l'embedding d'une zone masquée, peut-on en **ressortir la forme d'onde** ?

Protocole B (`jepa/decode.py`) : on entraîne un décodeur MLP par-token comme **lecteur neutre**
sur les VRAIS embeddings cibles (encodeur EMA gelé), on le **gèle**, puis on l'applique à la
prédiction du predictor. Deux lectures aux positions masquées (fold 10) :
- **D(z_tgt)** = borne haute (ce qu'un embedding parfait laisse décoder),
- **D(pred)** = ce que la *prédiction* du JEPA préserve réellement.

R² = 1 − MSE (signal z-normé, variance ≈ 1 → R²=0 ≡ prédire zéro). Le contrôle `--random-init`
décode un JEPA non entraîné : si `D(pred)` du JEPA ne bat pas le contrôle, la prédiction
n'apporte rien. Cellule **idempotente**.

In [ ]:
import subprocess, sys, json
from pathlib import Path

CKPT = RUN_DIR / 'ckpt_e99.pt'
assert CKPT.exists(), sorted(p.name for p in RUN_DIR.glob('*.pt'))
DEC = DRIVE_ROOT / 'runs' / 'decode'

def run_decode(arm):
    out = DEC / arm / 'dec.json'
    if out.exists():
        print(f'  décodeur {arm} : déjà fait'); return
    out.parent.mkdir(parents=True, exist_ok=True)
    src = ['--random-init'] if arm == 'random' else ['--ckpt', str(CKPT)]
    cmd = [sys.executable, '-m', 'jepa.decode', *src, '--plots', '4',
           '--out', str(out), '--workers', '2']
    print(f'\n=== décodeur {arm} ===', flush=True)
    assert subprocess.run(cmd).returncode == 0, f'échec décodeur {arm}'

for arm in ('jepa', 'random'):
    run_decode(arm)
print('\ndécodeur terminé')

### Résultats & tracés du décodeur

In [ ]:
import json, pandas as pd
from IPython.display import Image, display

rows = []
for arm in ('jepa', 'random'):
    p = DEC / arm / 'dec.json'
    if p.exists():
        m = json.load(open(p))['masked']
        rows.append({'arm': arm,
                     'R2 borne haute D(z_tgt)': m['tgt']['r2'],
                     'R2 prédiction D(pred)': m['pred']['r2']})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print('\n(R2 -> 1 = tracé parfaitement récupéré ; 0 = pas mieux que prédire zéro)')

# Tracés JEPA : vrai (noir) vs prédiction décodée (rouge) sur les zones masquées.
print('\nReconstructions JEPA (zones masquées) :')
for i in range(4):
    png = DEC / 'jepa' / f'recon_{i}.png'
    if png.exists():
        display(Image(str(png)))